# 18 — Paper Figures: New Insights (Robustness and Performance Diagram)

Two figures that go beyond notebooks 14-17 rather than restyling them.

**(a) Is classical-beats-generative universal, or driven by one classifier?**
Small multiples, one panel per classifier, same four algorithms in each.
Verified directly from `results/`: classical (ADASYN/SMOTE) beats
generative (TimeGAN/Diffusion) for **all four** classifiers with no
exceptions -- but the *size* of the gap is not uniform. GRU and PatchTST
show a large, decisive gap; SVM's two best algorithms are ADASYN and
SMOTE but the margin over the worse generative method is much smaller;
InceptionTime is weak across the board regardless of algorithm.

**(b) A performance diagram.** This project has been reporting TSS, FAR,
recall and bias as separate numbers across separate figures. A
performance diagram (Roebber 2009) plots probability of detection
against success ratio (1-FAR) with CSI shaded in the background and
constant-bias lines radiating from the origin -- every one of those
metrics in a single picture. Because this problem's false-alarm ratio
never drops below ~0.90 at any classifier or algorithm, the standard
0-1 x 0-1 diagram would crush all 16 points into one corner; this is a
**zoomed** diagram (a standard adjustment for rare-event verification),
success ratio 0-0.10.

**Reads:** `./results/*.txt`. **Requires:** notebook 14 run first.
**Writes:** `./paper/figures/robustness.pdf`/`.png` and
`./paper/figures/performance_diagram.pdf`/`.png`.


## 1. Robustness — Load and Preview

In [ ]:
AUG_ORDER = ["ADASYN", "SMOTE", "TimeGAN", "Diffusion"]
AUG_KEYS  = {"ADASYN": "tomek_rus500_adasyn500", "SMOTE": "tomek_rus500_smote500",
            "TimeGAN": "timegan_500", "Diffusion": "diffusion_500"}
AUG_FAMILY = {"ADASYN": "classical", "SMOTE": "classical",
             "TimeGAN": "generative", "Diffusion": "generative"}
FAMILY_COLOR = {"classical": TEAL, "generative": VERM}

CLF_ORDER = ["GRU", "PatchTST", "SVM", "InceptionTime"]
CLF_SLUG = {v: k for k, v in CLASSIFIER_LABELS.items()}

print(f"{'classifier':<14}", *[f"{a:>10}" for a in AUG_ORDER], "  classical wins?")
for clf in CLF_ORDER:
    means = {a: load_col(CLF_SLUG[clf], AUG_KEYS[a]).mean() for a in AUG_ORDER}
    win = max(means["ADASYN"], means["SMOTE"]) > max(means["TimeGAN"], means["Diffusion"])
    print(f"{clf:<14}", *[f"{means[a]:>10.3f}" for a in AUG_ORDER], f"  {win}")


## Figure — Is the Augmentation Result Universal?

In [ ]:
# FIG -- per-classifier robustness of the classical-vs-generative gap
fig, axes = plt.subplots(1, 4, figsize=(7.16, 2.15), sharey=True)

for ax, clf in zip(axes, CLF_ORDER):
    slug = CLF_SLUG[clf]
    for i, a in enumerate(AUG_ORDER):
        v = load_col(slug, AUG_KEYS[a])
        col = FAMILY_COLOR[AUG_FAMILY[a]]
        ax.bar(i, v.mean(), 0.62, color=col, edgecolor="none", zorder=3)
        ax.scatter(np.full_like(v, i, dtype=float), v, s=5, color="0.15", lw=0, zorder=5)
    ax.set_xticks(range(len(AUG_ORDER)))
    ax.set_xticklabels(AUG_ORDER, rotation=38, ha="right", fontsize=5.7)
    for tick, a in zip(ax.get_xticklabels(), AUG_ORDER):
        tick.set_color(FAMILY_COLOR[AUG_FAMILY[a]])
    ax.set_ylim(0, 0.75)
    finish(ax, ylab="Test TSS" if clf == "GRU" else None)
    ax.set_title(clf, loc="left", fontsize=6.8, pad=4)

fig.suptitle("Classical beats generative for every classifier -- but not by the same margin",
            fontsize=7.2, y=1.06, x=0.51)
fig.subplots_adjust(left=0.075, right=0.995, top=0.82, bottom=0.30, wspace=0.15)
fig.savefig(f"{FIG_DIR}/robustness.pdf", bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/robustness.png", dpi=340, bbox_inches="tight")
plt.show()
print("Saved robustness.pdf/png")


## 2. Performance Diagram — Load and Preview

In [ ]:
# ══════════════════════════════════════════════════════════════
# POD (recall) and success ratio (1 - FAR) for every classifier x
# augmentation-algorithm combination -- 16 points.
# ══════════════════════════════════════════════════════════════

points = []
for clf in CLF_ORDER:
    slug = CLF_SLUG[clf]
    for a in AUG_ORDER:
        key = AUG_KEYS[a]
        pod = load_col(slug, key, COLUMN["recall"]).mean()
        far = load_col(slug, key, COLUMN["far"]).mean()
        sr = 1 - far
        points.append(dict(classifier=clf, algorithm=a, family=AUG_FAMILY[a],
                           pod=pod, sr=sr))

print(f"{'classifier':<14} {'algorithm':<10} {'POD':>7} {'SR':>7}")
for p in points:
    print(f"{p['classifier']:<14} {p['algorithm']:<10} {p['pod']:>7.3f} {p['sr']:>7.3f}")


## Figure — Performance Diagram: Everything in One View

In [ ]:
# FIG -- zoomed performance diagram: POD vs success ratio,
# CSI shaded in the background, bias as diagonals from the origin
fig, ax = plt.subplots(figsize=(3.6, 3.4))

SR_MAX = 0.10
sr_grid = np.linspace(0.002, SR_MAX, 400)
pod_grid = np.linspace(0.002, 1.0, 400)
SR, POD = np.meshgrid(sr_grid, pod_grid)
CSI = 1.0 / (1.0 / POD + 1.0 / SR - 1.0)
CSI = np.clip(CSI, 0, 1)

cf = ax.contourf(SR, POD, CSI, levels=np.linspace(0, 0.12, 13),
                 cmap="Greens", alpha=0.55, extend="max", zorder=1)
cbar = fig.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("CSI", fontsize=6.4)
cbar.ax.tick_params(labelsize=5.6)

for b in [1, 2, 5, 10, 20, 50]:
    sr_line = np.linspace(0.0005, SR_MAX, 50)
    pod_line = b * sr_line
    mask = pod_line <= 1.0
    ax.plot(sr_line[mask], pod_line[mask], color="0.55", lw=0.5, ls=(0, (2, 2)), zorder=2)
    if (pod_line <= 1.0).any():
        xi = sr_line[mask][-1]
        yi = pod_line[mask][-1]
        ax.text(xi, min(yi, 0.99), f"{b}", fontsize=5.0, color="0.45",
                ha="left", va="bottom", zorder=2)

MARKERS = {"GRU": "o", "PatchTST": "s", "SVM": "^", "InceptionTime": "D"}
for p in points:
    ax.scatter(p["sr"], p["pod"], marker=MARKERS[p["classifier"]],
              s=26, color=FAMILY_COLOR[p["family"]], edgecolor="white",
              linewidth=0.5, zorder=4)

ax.set_xlim(0, SR_MAX)
ax.set_ylim(0, 1.0)
finish(ax, ylab="POD (recall)", xlab="Success ratio (1 $-$ FAR)", grid=None)
ax.set_title("All 16 classifier x algorithm points", loc="left", fontsize=7.0, pad=6)

fam_handles = [Patch(facecolor=TEAL, label="classical"),
              Patch(facecolor=VERM, label="generative")]
clf_handles = [plt.Line2D([0], [0], marker=m, linestyle="", color="0.25",
                          markersize=4.5, label=c) for c, m in MARKERS.items()]
leg1 = ax.legend(handles=fam_handles, loc="upper left", frameon=False,
                 fontsize=5.6, handlelength=1.0, handletextpad=0.4,
                 borderpad=0.1, labelspacing=0.2, bbox_to_anchor=(0.0, 1.0))
ax.add_artist(leg1)
ax.legend(handles=clf_handles, loc="upper left", frameon=False, fontsize=5.6,
          handlelength=1.0, handletextpad=0.4, borderpad=0.1, labelspacing=0.2,
          bbox_to_anchor=(0.0, 0.80))
ax.text(0.985, 0.03, "dashed lines: bias", transform=ax.transAxes, fontsize=5.0,
        color="0.45", ha="right", style="italic")

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/performance_diagram.pdf")
fig.savefig(f"{FIG_DIR}/performance_diagram.png", dpi=340)
plt.show()
print("Saved performance_diagram.pdf/png")
print("Every point sits at bias >> 1 (all 16 classifier x algorithm combinations "
      "over-forecast) and success ratio under 0.08 for all of them: the extreme "
      "false-alarm ratio is a property of the problem, not any one method.")
